# Notebook 2 – Data Type Handling

## 1. Identifying data types
- **Numerical:** counts and measurements such as age, price, or temperature.
- **Categorical:** labels such as city, product type, or customer segment. Categories may be nominal or ordinal.
- **Boolean:** true/false indicators, often stored as `bool`, `0/1`, or text.
- **Date/time:** timestamps, dates, durations, and time zones.
- **String/text:** names, descriptions, identifiers, and raw imported values.

A column stored as an integer is not necessarily a quantity. A postal code or customer ID is usually an identifier, not a numeric measurement.

## 2. Inspecting a DataFrame

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": ["25", " 31 ", "unknown", "42"],
    "join_date": ["2024-01-05", "2024/02/10", "not-a-date", "2024-04-01"],
    "is_active": ["True", "False", "yes", "no"],
    "customer_id": [1001, 1002, 1003, 1004],
    "segment": ["1", "2", "1", "3"],
    "salary": [25000, 31000, 28000, 42000]
})

display(df)
print(df.dtypes)
df.info()

## 3. String → Numeric with `to_numeric()`
Use `errors="coerce"` when invalid strings should become `NaN` so that they can be investigated. Do not silently ignore how many conversions failed.

In [ ]:
df["age_numeric"] = pd.to_numeric(df["age"].str.strip(), errors="coerce")
display(df[["age", "age_numeric"]])

invalid_age = df[df["age"].notna() & df["age_numeric"].isna()]
print("Values that failed conversion:")
display(invalid_age[["age"]])

## 4. String → Date/Time with `to_datetime()`
Specify a known format when possible. Use `errors="coerce"` to flag malformed dates. Consider time zones when timestamps come from different locations.

In [ ]:
df["join_date_parsed"] = pd.to_datetime(df["join_date"], errors="coerce")
display(df[["join_date", "join_date_parsed"]])

invalid_dates = df[df["join_date"].notna() & df["join_date_parsed"].isna()]
print("Invalid date values:")
display(invalid_dates[["join_date"]])

## 5. `astype()` conversions
`astype()` is useful when values are already compatible with the requested type. It can raise an error when a value cannot be converted. For messy imported values, clean/validate first or use `to_numeric()`/`to_datetime()` with explicit error handling.

In [ ]:
# Integer → Float
df["salary_float"] = df["salary"].astype("float64")

# Numeric codes → Categorical labels (only if these codes represent categories)
df["segment_category"] = df["segment"].astype("category")

# Safe Boolean conversion from known text values
bool_map = {"true": True, "yes": True, "false": False, "no": False}
df["is_active_bool"] = (
    df["is_active"].str.strip().str.lower().map(bool_map).astype("boolean")
)

display(df[["salary", "salary_float", "segment", "segment_category",
            "is_active", "is_active_bool"]])

## 6. Handling invalid conversions
1. Preserve the original column until conversion is validated.
2. Convert using an explicit rule.
3. Count and inspect values that became missing.
4. Decide whether to correct, reject, mark missing, or keep as text.
5. Record the rule and re-check the resulting type and value range.

Avoid replacing every failed conversion with zero: zero may be a real value and would distort statistics.

In [ ]:
raw_values = pd.Series(["100", "250.5", "unknown", "", " 75 "], name="raw")
cleaned = raw_values.astype("string").str.strip().replace("", pd.NA)
numeric = pd.to_numeric(cleaned, errors="coerce")

audit = pd.DataFrame({"original": raw_values, "cleaned": cleaned, "numeric": numeric})
audit["conversion_failed"] = audit["cleaned"].notna() & audit["numeric"].isna()
display(audit)
print("Failed conversions:", int(audit["conversion_failed"].sum()))

## 7. Detecting incorrect data types
Examples of warning signs:
- Numbers imported as `object`/string.
- Dates stored as plain text.
- Boolean values represented inconsistently (`Y`, `yes`, `1`, `TRUE`).
- IDs treated as measurements.
- Numeric codes mistakenly treated as continuous values.
- A column contains mixed units or formats.

## 8. Why correct data types matter
- Arithmetic and statistical summaries behave correctly.
- Dates can be sorted, filtered, and used to derive durations.
- Categorical encoding can be applied appropriately.
- Memory usage can be reduced with suitable types.
- Invalid records become easier to detect.
- Model preprocessing can use the right transformations.

## 9. Practice exercises
1. Convert a column containing `"10"`, `"20"`, and `"unknown"` to numeric and report failed conversions.
2. Parse a date column that uses `DD-MM-YYYY`.
3. Explain why a PIN code should usually be stored as a string.
4. Convert a `0/1` flag to Boolean only after verifying its meaning.
5. Identify a column that looks numeric but is actually categorical.